In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, r2_score, mean_absolute_error, mean_squared_error

# Root of the repo
REPO_ROOT = Path("/endosome/work/biophysics/s438168/DeepFISIK")
LOCAL_SRC = REPO_ROOT / "src"

# Make subprocesses import the local code
env = os.environ.copy()
env["PYTHONPATH"] = str(LOCAL_SRC)

# Choose one:
# task = "classification"
# task = "dimer"
# task = "trimer"
# task = "tetramer"
task = "dimer"

run_date = "run"
batch_size = 1
seed = 50
time_step = 0.1

save_dir = REPO_ROOT / "runs" / "PureSimulations" / task
output_dir = save_dir / "inference"

if task == "classification":
    dataset_roots = [
        REPO_ROOT / "Datasets" / "Images" / "MonomerDataset",
        REPO_ROOT / "Datasets" / "Images" / "DimerDataset",
        REPO_ROOT / "Datasets" / "Images" / "TrimerDataset",
        REPO_ROOT / "Datasets" / "Images" / "TetramerDataset",
    ]
    predict_script = REPO_ROOT / "scripts" / "predict_classification.py"
    checkpoint = REPO_ROOT / "models" / "Images" / "classification" / "ImageClassificationModel.pt"
else:
    dataset_roots = [
        #REPO_ROOT / "Datasets" / "PureSimulations" / f"{task.capitalize()}Dataset"
        REPO_ROOT / "Datasets" / "PureSimulations" / "testDimer"
    ]
    predict_script = REPO_ROOT / "scripts" / f"predict_{task}s.py"
    # checkpoint_map = {
    #     "dimer": REPO_ROOT / "models" / "Images" / "dimers" / "ImageDimerModel.pt",
    #     "trimer": REPO_ROOT / "models" / "PureSimulations" / "trimers" / "PureSimulationsTrimerModel.pt",
    #     "tetramer": REPO_ROOT / "models" / "PureSimulations" / "tetramers" / "PureSimulationsTetramerModel.pt",
    # }
    # checkpoint = checkpoint_map[task]
    checkpoint = REPO_ROOT/ "models" /"PureSimulations"/ "dimers" / "PureSimulationsDimerModel.pt"

print("Task:", task)
print("Predict script:", predict_script)
print("Checkpoint:", checkpoint)
print("Dataset roots:", [str(p) for p in dataset_roots])
print("Output dir:", output_dir)

In [ ]:
predict_cmd = [
    sys.executable, str(predict_script),
    "--date", run_date,
    "--output-dir", str(output_dir),
    "--batch-size", str(batch_size),
    "--seed", str(seed),
    "--checkpoint", str(checkpoint),
    "--dataset-roots", *map(str, dataset_roots),
    "--split", "all"
]

if task != "classification":
    predict_cmd += ["--time-step", str(time_step)]

print(" ".join(predict_cmd))
subprocess.run(predict_cmd, cwd=REPO_ROOT, env=env, check=True)

In [ ]:
predictions_csv = output_dir / f"predictions_{run_date}.csv"
metrics_csv = output_dir / f"metrics_{run_date}.csv"

df = pd.read_csv(predictions_csv)
display(df.head())
print(df.columns)

y_true = df["true_class"]
y_pred = df["pred_class"]

labels = ["Monomer", "Dimer", "Trimer", "Tetramer"]
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(6, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, values_format="d", cmap="Blues", colorbar=False)
plt.title("DeepFISIK Classification Confusion Matrix")
plt.tight_layout()
plt.show()

if metrics_csv.exists():
    display(pd.read_csv(metrics_csv))

In [ ]:
predictions_csv = output_dir / task / f"predictions_{run_date}.csv"
metrics_csv = output_dir / task / f"metrics_{run_date}.csv"

df = pd.read_csv(predictions_csv)
display(df.head())
print(df.columns)

if metrics_csv.exists():
    metrics_df = pd.read_csv(metrics_csv)
    display(metrics_df)

In [ ]:
def target_pairs(frame: pd.DataFrame):
    pairs = []
    for col in frame.columns:
        if col.startswith("true_"):
            name = col[len("true_"):]
            pred_col = f"pred_{name}"
            if pred_col in frame.columns:
                pairs.append((name, col, pred_col))
    return pairs

def summarize_targets(frame: pd.DataFrame, pairs):
    rows = []
    for name, true_col, pred_col in pairs:
        y_true = frame[true_col].astype(float)
        y_pred = frame[pred_col].astype(float)
        rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
        mean_true = float(np.mean(np.abs(y_true)))

        rows.append({
            "target": name,
            "r2": float(r2_score(y_true, y_pred)),
            "mae": float(mean_absolute_error(y_true, y_pred)),
            "rmse": rmse,
            "normalized_rmse": float(rmse / mean_true) if mean_true != 0 else np.nan,
            "num_samples": int(len(frame)),
        })
    return pd.DataFrame(rows)

def plot_regression_properties(frame: pd.DataFrame, pairs, title_prefix: str):
    n = len(pairs)
    if n == 0:
        print("No true_/pred_ columns found.")
        return

    cols = 2
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 5 * rows))
    axes = np.atleast_1d(axes).ravel()

    for ax, (name, true_col, pred_col) in zip(axes, pairs):
        y_true = frame[true_col].astype(float).to_numpy()
        y_pred = frame[pred_col].astype(float).to_numpy()

        ax.scatter(y_true, y_pred, s=18)
        lo = min(np.min(y_true), np.min(y_pred))
        hi = max(np.max(y_true), np.max(y_pred))
        ax.plot([lo, hi], [lo, hi])
        ax.set_title(f"{name} | R2={r2_score(y_true, y_pred):.3f}")
        ax.set_xlabel("True")
        ax.set_ylabel("Predicted")

    for ax in axes[len(pairs):]:
        ax.axis("off")

    fig.suptitle(f"{title_prefix} regression properties", y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
pairs = target_pairs(df)
summary_df = summarize_targets(df, pairs)
display(summary_df)
plot_regression_properties(df, pairs, task)